In [1]:
import glob
import os
import zipfile
import rasterio
from rasterio.merge import merge
import numpy as np

## Unificar MDT

In [2]:
pasta_zips = '../dados/mdt/sao_lor_mat_zip/'
pasta_extraidos = '../dados/mdt/sao_lor_mat_tif/'
caminho_mde_unificado = '../dados/rasters/mdt_sao_lourenco_pe3d.tif'

os.makedirs(pasta_extraidos, exist_ok=True)

for file in glob.glob(os.path.join(pasta_zips, "*.zip")):
    with zipfile.ZipFile(file, 'r') as zip_ref:
        zip_ref.extractall(pasta_extraidos)

tifs_para_unir = glob.glob(os.path.join(pasta_extraidos, "**/*.tif"), recursive=True)

rasters_abertos = [rasterio.open(tif) for tif in tifs_para_unir]

mosaico_array, out_trans = merge(rasters_abertos)

perfil = rasters_abertos[0].profile.copy()
perfil.update({
    "height": mosaico_array.shape[1],
    "width": mosaico_array.shape[2],
    "transform": out_trans,
    "count": 1,
    "dtype": rasterio.float32
})

with rasterio.open(caminho_mde_unificado, "w", **perfil) as dst:
    dst.write(mosaico_array[0].astype(np.float32), 1)

for r in rasters_abertos:
    r.close()

print(f"MDT unificado salvo com sucesso em: {caminho_mde_unificado}")

MDT unificado salvo com sucesso em: ../dados/rasters/mdt_sao_lourenco_pe3d.tif


## Reduzir resolução do MDT

In [3]:
import rasterio
from rasterio.enums import Resampling

def reduzir_resolucao_mdt(caminho_entrada, caminho_saida, fator_escala=5):
    with rasterio.open(caminho_entrada) as src:
        nova_largura = int(src.width / fator_escala)
        nova_altura = int(src.height / fator_escala)

        nova_transform = src.transform * src.transform.scale(
            (src.width / nova_largura),
            (src.height / nova_altura)
        )

        dados_reduzidos = src.read(
            1,
            out_shape=(nova_altura, nova_largura),
            resampling=Resampling.bilinear
        )

        perfil = src.profile.copy()
        perfil.update({
            "driver": "GTiff",
            "height": nova_altura,
            "width": nova_largura,
            "transform": nova_transform,
            "compress": "lzw"
        })

        with rasterio.open(caminho_saida, "w", **perfil) as dst:
            dst.write(dados_reduzidos, 1)

    print(f"MDT salvo, Dimensão reduzida de ({src.height}, {src.width}) para ({nova_altura}, {nova_largura}).")

# mdt_original = '../dados/camadas_rasters/mdt_recife_pe3d.tif'
# mdt_leve = '../dados/camadas_rasters/mdt_recife_pe3d_10m.tif'

mdt_original = '../dados/rasters/mdt_sao_lourenco_pe3d.tif'
mdt_leve = '../dados/rasters/mdt_sao_lourenco_pe3d_10m.tif'

reduzir_resolucao_mdt(mdt_original, mdt_leve, fator_escala=10)

MDT salvo, Dimensão reduzida de (18643, 31148) para (1864, 3114).
